# CS231 - Thành viên 1: Trích xuất đặc trưng Thủ công (Hand-crafted) & Máy học (SVM / XGBoost)


In [ ]:
import sys, os
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.member1_classical.feature_extractor import WasteFeatureExtractor

extractor = WasteFeatureExtractor()
print("Initialized WasteFeatureExtractor (HSV 3D + LBP + HOG)")


### 1. Trích xuất và trực quan hóa 3 nhóm đặc trưng trên cùng 1 ảnh


In [ ]:
import glob
sample_files = glob.glob('../data/standard_split/train/metal/*.*')
if sample_files:
    bgr = cv2.imread(sample_files[0])
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    features, viz_blended = extractor.extract_features(rgb, return_viz=True)
    
    print(f"Tổng số chiều vector đặc trưng kết hợp: {len(features)}")
    
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(rgb); ax[0].set_title("Ảnh gốc (Metal Can)"); ax[0].axis('off')
    ax[1].imshow(viz_blended); ax[1].set_title("Biểu diễn đặc trưng HOG Gradients"); ax[1].axis('off')
    plt.tight_layout()
    plt.show()


### 2. Huấn luyện Pipeline (StandardScaler + PCA + SVM/XGBoost)


In [ ]:
from src.member1_classical.train_ml import train_classical_pipeline

# Huấn luyện pipeline và lưu trọng số vào weights/classical_svm_pipeline.pkl
train_classical_pipeline(config_path='../configs/config.yaml', model_choice='svm')


### 3. Kiểm thử suy luận với ClassicalPredictor


In [ ]:
from src.member1_classical.predictor_ml import ClassicalPredictor

predictor = ClassicalPredictor(weight_path='../weights/classical_svm_pipeline.pkl')
if sample_files:
    result = predictor.predict(cv2.resize(rgb, (224, 224)))
    print("Dự đoán:", result['class_name'])
    print("Độ tự tin:", result['confidence'])
    print("Độ trễ:", result['latency_ms'], "ms")
    print("Thùng rác kiến nghị:", result['action_bin'])
